# Fabric Dataset Refresh Monitoring

Collects **dataset refresh operations** and **metadata** from Fabric REST APIs and sends to Azure Log Analytics.

In [ ]:
# === Framework Integration for Dataset Refresh Monitoring ===
import sys
import os

# Add the framework to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Import refactored framework components
from fabricla_connector.config import get_config
from fabricla_connector.api import FabricAPIClient, get_fabric_token
from fabricla_connector.collectors import DatasetRefreshCollector
from fabricla_connector.ingestion import post_rows_to_dcr, AzureMonitorIngestionClient
from fabricla_connector.mappers import DatasetRefreshMapper, DatasetMetadataMapper
from fabricla_connector.utils import iso_now, within_lookback_minutes
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

# Load configuration
load_dotenv()
config = get_config()

print("✅ FabricLA Connector framework loaded for dataset refresh monitoring")
print("✅ Using refactored collectors, mappers, and ingestion modules")

In [ ]:
# === Parameters (mark this as a parameter cell in Fabric) ===
import os
from dotenv import load_dotenv

load_dotenv()

# Multi-workspace monitoring
# Empty list = monitor ALL accessible workspaces
# Specific workspaces = ["workspace-id-1", "workspace-id-2"]
workspace_ids = (
    [os.getenv("FABRIC_WORKSPACE_ID")] if os.getenv("FABRIC_WORKSPACE_ID") else []
)

# Dataset filtering — leave empty to monitor all datasets in workspaces
dataset_ids = []  # or: ["8c4003da-3254-4a1a-b07e-ac62cb86b5cf"]

# Lookback window for refresh history (max depends on Power BI retention — typically 60 days)
lookback_hours = 168  # 7 days default; increase up to ~1440 (60 days)

# Azure Monitor ingestion settings — must match your DCE / DCR deployment
dce_endpoint     = os.getenv("AZURE_MONITOR_DCE_ENDPOINT")          # full URL
dcr_immutable_id = os.getenv("AZURE_MONITOR_DCR_IMMUTABLE_ID")  # DCR immutableId
stream_dataset_refresh  = "Custom-FabricDatasetRefresh_CL"
stream_dataset_metadata = "Custom-FabricDatasetMetadata_CL"

tenant_id         = os.getenv("FABRIC_TENANT_ID")
client_id         = os.getenv("FABRIC_APP_ID")
client_secret_env = os.getenv("FABRIC_APP_SECRET")

use_key_vault        = False
use_managed_identity = False
key_vault_uri        = os.getenv("AZURE_KEY_VAULT_URI", "https://<your-keyvault-name>.vault.azure.net/")
key_vault_secret_name = os.getenv("AZURE_KEY_VAULT_SECRET_NAME", "FabricServicePrincipal")

if not all([tenant_id, client_id, dce_endpoint, dcr_immutable_id]):
    missing = []
    if not tenant_id:         missing.append("FABRIC_TENANT_ID")
    if not client_id:         missing.append("FABRIC_APP_ID")
    if not dce_endpoint:      missing.append("AZURE_MONITOR_DCE_ENDPOINT")
    if not dcr_immutable_id:  missing.append("AZURE_MONITOR_DCR_IMMUTABLE_ID")
    print(f"❌ Missing: {', '.join(missing)}")
else:
    print("✅ Environment variables loaded")

print(f"Workspace mode: {'Specific workspaces' if workspace_ids else 'Auto-discovery'}")
print(f"Dataset mode: {'Specific datasets' if dataset_ids else 'All datasets'}")
print(f"Lookback: {lookback_hours} hours")


In [ ]:
# === Define main functions ===
import os, json, time, datetime as dt
import requests
from typing import List, Dict, Any

def get_secret_from_kv(vault_uri, secret_name, tenant_id=None, client_id=None, client_secret=None, use_managed_identity=False):
    try:
        from azure.keyvault.secrets import SecretClient
        if use_managed_identity:
            from azure.identity import ManagedIdentityCredential
            credential = ManagedIdentityCredential()
        else:
            from azure.identity import ClientSecretCredential
            credential = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
        client = SecretClient(vault_url=vault_uri, credential=credential)
        return client.get_secret(secret_name).value
    except Exception as e:
        print(f"[KeyVault] Failed: {e}")
        return None

FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"
MONITOR_SCOPE = "https://monitor.azure.com/.default"
FABRIC_API = "https://api.fabric.microsoft.com/v1"

def acquire_token_client_credentials(tenant, client_id, client_secret, scope):
    import msal
    authority = f"https://login.microsoftonline.com/{tenant}"
    app = msal.ConfidentialClientApplication(client_id, authority=authority, client_credential=client_secret)
    result = app.acquire_token_for_client(scopes=[scope])
    if "access_token" not in result:
        raise RuntimeError(f"Failed to get token for {scope}: {result}")
    print(f"✅ Token acquired for {scope}")
    return result["access_token"]

def iso_now():
    return dt.datetime.utcnow().replace(tzinfo=dt.timezone.utc).isoformat().replace("+00:00", "Z")

def parse_iso(s):
    if not s:
        return None
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    parsed = dt.datetime.fromisoformat(s)
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=dt.timezone.utc)
    return parsed

def list_workspace_datasets(workspace_id: str, token: str) -> List[Dict[str, Any]]:
    """
    List Semantic Models (datasets) in a workspace.
    API: GET /v1/workspaces/{workspaceId}/items?type=SemanticModel
    """
    url = f"{FABRIC_API}/workspaces/{workspace_id}/items"
    headers = {"Authorization": f"Bearer {token}"}
    items, continuation_token = [], None

    while True:
        params = {"type": "SemanticModel"}
        if continuation_token:
            params["continuationToken"] = continuation_token
        try:
            r = requests.get(url, headers=headers, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()
            items.extend(data.get("value", []))
            continuation_token = data.get("continuationToken")
            if not continuation_token:
                break
        except Exception as e:
            print(f"❌ Failed to list datasets for workspace {workspace_id}: {e}")
            break

    # Normalise to a consistent shape (displayName → name for downstream compat)
    return [
        {
            "id": item.get("id"),
            "name": item.get("displayName", item.get("name", "Unknown")),
            "type": item.get("type"),
            "description": item.get("description"),
            "workspaceId": workspace_id,
        }
        for item in items
    ]

def get_dataset_refresh_history(workspace_id: str, dataset_id: str, token: str) -> List[Dict[str, Any]]:
    """
    Get refresh job instances for a Semantic Model via the Job Scheduler API.
    API: GET /v1/workspaces/{workspaceId}/items/{datasetId}/jobs/instances
    Filters to DefaultDatasetRefresh job type.
    """
    url = f"{FABRIC_API}/workspaces/{workspace_id}/items/{dataset_id}/jobs/instances"
    headers = {"Authorization": f"Bearer {token}"}
    instances, continuation_token = [], None

    while True:
        params = {}
        if continuation_token:
            params["continuationToken"] = continuation_token
        try:
            r = requests.get(url, headers=headers, params=params, timeout=60)
            if r.status_code == 404:
                return []  # item has no job history
            r.raise_for_status()
            data = r.json()
            for inst in data.get("value", []):
                job_type = inst.get("jobType", "DefaultDatasetRefresh")
                if job_type == "DefaultDatasetRefresh":
                    instances.append(inst)
            continuation_token = data.get("continuationToken")
            if not continuation_token:
                break
        except Exception as e:
            print(f"❌ Failed to get refresh history for dataset {dataset_id}: {e}")
            break

    return instances

def get_dataset_metadata(workspace_id: str, dataset_id: str, token: str) -> Dict[str, Any]:
    """
    Get item metadata for a Semantic Model.
    API: GET /v1/workspaces/{workspaceId}/items/{datasetId}
    """
    url = f"{FABRIC_API}/workspaces/{workspace_id}/items/{dataset_id}"
    headers = {"Authorization": f"Bearer {token}"}
    try:
        r = requests.get(url, headers=headers, timeout=60)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"❌ Failed to get metadata for dataset {dataset_id}: {e}")
        return {}

def map_dataset_refresh(workspace_id: str, dataset_id: str, dataset_name: str, instance: Dict[str, Any]) -> Dict[str, Any]:
    """Map a Job Scheduler instance to the FabricDatasetRefresh_CL schema."""
    start_time = instance.get("startTimeUtc")
    end_time   = instance.get("endTimeUtc")

    duration_ms = None
    if start_time and end_time:
        try:
            start_dt = parse_iso(start_time)
            end_dt   = parse_iso(end_time)
            if start_dt and end_dt:
                duration_ms = int((end_dt - start_dt).total_seconds() * 1000)
        except Exception:
            pass

    failure = instance.get("failureReason") or {}
    return {
        "TimeGenerated":   end_time or start_time or iso_now(),
        "WorkspaceId":     workspace_id,
        "DatasetId":       dataset_id,
        "DatasetName":     dataset_name,
        "RefreshId":       instance.get("id"),
        "RefreshType":     instance.get("invokeType"),   # Scheduled | Manual | etc.
        "Status":          instance.get("status"),       # Completed | Failed | InProgress | etc.
        "StartTime":       start_time,
        "EndTime":         end_time,
        "DurationMs":      duration_ms,
        "JobType":         instance.get("jobType"),
        "RootActivityId":  instance.get("rootActivityId"),
        "ErrorCode":       failure.get("errorCode") if isinstance(failure, dict) else None,
        "ErrorMessage":    failure.get("message")   if isinstance(failure, dict) else str(failure) if failure else None,
    }

def map_dataset_metadata(workspace_id: str, dataset: Dict[str, Any]) -> Dict[str, Any]:
    """Map a Fabric items API response to the FabricDatasetMetadata_CL schema."""
    return {
        "TimeGenerated": iso_now(),
        "WorkspaceId":   workspace_id,
        "DatasetId":     dataset.get("id"),
        "DatasetName":   dataset.get("displayName") or dataset.get("name"),
        "Description":   dataset.get("description"),
        "Type":          dataset.get("type"),
    }

def list_accessible_workspaces(token: str) -> List[Dict[str, Any]]:
    url = f"{FABRIC_API}/workspaces"
    headers = {"Authorization": f"Bearer {token}"}
    items, continuation_token = [], None
    while True:
        params = {}
        if continuation_token:
            params["continuationToken"] = continuation_token
        try:
            r = requests.get(url, headers=headers, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()
            items.extend(data.get("value", []))
            continuation_token = data.get("continuationToken")
            if not continuation_token:
                break
        except Exception as e:
            print(f"❌ Failed to list workspaces: {e}")
            break
    return items

def post_rows_to_dcr(endpoint_host: str, dcr_id: str, stream_name: str, rows: List[Dict[str, Any]], monitor_token: str):
    if not rows:
        return {"sent": 0, "batches": 0}

    MAX_BYTES = 950_000
    batch, batches, sent, size = [], 0, 0, 2

    def flush():
        nonlocal batches, sent, batch, size
        if not batch:
            return
        url = f"{endpoint_host.rstrip('/')}/dataCollectionRules/{dcr_id}/streams/{stream_name}?api-version=2023-01-01"
        headers = {"Authorization": f"Bearer {monitor_token}", "Content-Type": "application/json"}
        resp = requests.post(url, headers=headers, data=json.dumps(batch), timeout=60)
        if resp.status_code >= 400:
            raise RuntimeError(f"Ingestion failed ({resp.status_code}): {resp.text[:500]}")
        batches += 1
        sent += len(batch)
        batch, size = [], 2

    for row in rows:
        s = len(json.dumps(row, separators=(",", ":")))
        if size + s + (1 if batch else 0) > MAX_BYTES:
            flush()
        batch.append(row)
        size += s + (1 if batch else 0)
    flush()
    return {"sent": sent, "batches": batches}

print("✅ Functions loaded")

In [ ]:
# === Main ===
client_secret = None

if client_secret_env:
    client_secret = client_secret_env
    print("✅ Using environment variable")
elif use_key_vault:
    if use_managed_identity:
        client_secret = get_secret_from_kv(key_vault_uri, key_vault_secret_name, use_managed_identity=True)
    else:
        temp_secret = os.getenv("FABRIC_APP_SECRET")
        if not temp_secret:
            raise RuntimeError("FABRIC_APP_SECRET required for Key Vault access")
        client_secret = get_secret_from_kv(key_vault_uri, key_vault_secret_name, tenant_id, client_id, temp_secret)

if not client_secret:
    raise RuntimeError("Client secret not found")

print("✅ Client secret resolved")

fabric_token  = acquire_token_client_credentials(tenant_id, client_id, client_secret, FABRIC_SCOPE)
monitor_token = acquire_token_client_credentials(tenant_id, client_id, client_secret, MONITOR_SCOPE)

# Enumerate accessible workspaces
print("\n🔍 Checking accessible workspaces...")
accessible_workspaces = list_accessible_workspaces(fabric_token)
print(f"Found {len(accessible_workspaces)} accessible workspace(s)")
for ws in accessible_workspaces[:10]:
    print(f"  - {ws.get('displayName', ws.get('name', 'Unknown'))} ({ws.get('id', 'No ID')})")
if len(accessible_workspaces) > 10:
    print(f"  ... and {len(accessible_workspaces) - 10} more")

now = dt.datetime.utcnow().replace(tzinfo=dt.timezone.utc)
cutoff_time = now - dt.timedelta(hours=lookback_hours)

print(f"\nCollecting dataset refresh data from last {lookback_hours} hours...")

refresh_rows  = []
metadata_rows = []
total_datasets  = 0
total_refreshes = 0

workspaces_to_process = workspace_ids if workspace_ids else [ws.get("id") for ws in accessible_workspaces]

for workspace_id in workspaces_to_process:
    print(f"\nProcessing workspace: {workspace_id}")

    # API: GET /v1/workspaces/{ws}/items?type=SemanticModel
    datasets = list_workspace_datasets(workspace_id, fabric_token)
    print(f"  Found {len(datasets)} Semantic Model(s)")

    for dataset in datasets:
        dataset_id   = dataset.get("id")
        dataset_name = dataset.get("name", "Unknown")

        if dataset_ids and dataset_id not in dataset_ids:
            continue

        print(f"  Processing: {dataset_name} ({dataset_id})")
        total_datasets += 1

        # Metadata: GET /v1/workspaces/{ws}/items/{id}
        detailed_metadata = get_dataset_metadata(workspace_id, dataset_id, fabric_token)
        metadata_rows.append(map_dataset_metadata(workspace_id, detailed_metadata or dataset))

        # Refresh history: GET /v1/workspaces/{ws}/items/{id}/jobs/instances (DefaultDatasetRefresh)
        instances = get_dataset_refresh_history(workspace_id, dataset_id, fabric_token)

        recent = [
            inst for inst in instances
            if parse_iso(inst.get("endTimeUtc") or inst.get("startTimeUtc")) and
               parse_iso(inst.get("endTimeUtc") or inst.get("startTimeUtc")) >= cutoff_time
        ]

        print(f"    Found {len(recent)} refresh job(s) in lookback window")
        total_refreshes += len(recent)

        for inst in recent:
            refresh_rows.append(map_dataset_refresh(workspace_id, dataset_id, dataset_name, inst))

print(f"\nCollection Summary:")
print(f"  Total Datasets:  {total_datasets}")
print(f"  Total Refreshes: {total_refreshes}")
print(f"  Metadata Records: {len(metadata_rows)}")
print(f"  Refresh Records:  {len(refresh_rows)}")

summary = {}

if metadata_rows:
    print("\nSending dataset metadata...")
    result = post_rows_to_dcr(dce_endpoint, dcr_immutable_id, stream_dataset_metadata, metadata_rows, monitor_token)
    summary["dataset_metadata"] = result

if refresh_rows:
    print("Sending refresh history...")
    result = post_rows_to_dcr(dce_endpoint, dcr_immutable_id, stream_dataset_refresh, refresh_rows, monitor_token)
    summary["dataset_refreshes"] = result

print("\n✅ Done!")
print(json.dumps(summary, indent=2))